<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/Modration_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

GPU available: True
Fri Oct  2 07:20:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [1]:
!pip install -q -U transformers accelerate hf_xet bitsandbytes opencv-python-headless gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, io, re, json, time, subprocess, mimetypes
from dataclasses import dataclass, field, asdict
from typing import Optional
import numpy as np, requests, torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoConfig, AutoProcessor, Llama4ForConditionalGeneration, BitsAndBytesConfig

model_id = "meta-llama/Llama-Guard-4-12B"
config = AutoConfig.from_pretrained(model_id)
if getattr(config.text_config, "attention_chunk_size", None) is None:
    config.text_config.attention_chunk_size = 8192

processor = AutoProcessor.from_pretrained(model_id)
model = Llama4ForConditionalGeneration.from_pretrained(
    model_id, config=config, device_map={"": 0},
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16),
)
model.eval()

# mask bug patch (idempotent)
import transformers.masking_utils as mu
import transformers.models.llama4.modeling_llama4 as ml4
if not getattr(mu, "_guard_patched", False):
    _real = mu.create_chunked_causal_mask
    def _patched(*a, _real=_real, **k):
        k.pop("block_sequence_ids", None)
        return _real(*a, **k)
    mu.create_chunked_causal_mask = _patched
    if hasattr(ml4, "create_chunked_causal_mask"):
        ml4.create_chunked_causal_mask = _patched
    for _n in dir(mu):
        _o = getattr(mu, _n)
        if isinstance(_o, dict):
            for _k, _v in list(_o.items()):
                if _v is _real:
                    _o[_k] = _patched
    mu._guard_patched = True
print(f"Loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

config.json:   0%|          | 0.00/2.79k [00:00<?, ?B/s]

[transformers] `rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1 and low_freq_factor=1.0


processor_config.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/989 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

Loaded. GPU memory: 9.7 GB


In [5]:
def _vision_dtype():
    for name, p in model.named_parameters():
        if "vision" in name and p.is_floating_point():
            return p.dtype
    return torch.bfloat16

def _run(text, images=(), max_new_tokens=20):
    if not images:
        inputs = processor.apply_chat_template(
            [{"role": "user", "content": [{"type": "text", "text": text or ""}]}],
            tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
        ).to(model.device)
    else:
        tok = getattr(processor, "fake_image_token", "<|image|>")
        prompt = processor.apply_chat_template(
            [{"role": "user", "content": [{"type": "text", "text": text or ""}]}],
            tokenize=False, add_generation_prompt=True)
        if tok not in prompt:
            m = re.search(r"User:\s*", prompt)
            if m is None:
                raise RuntimeError("Couldn't find 'User:' in prompt:\n" + prompt)
            prompt = prompt[:m.end()] + (tok + " ") * len(images) + prompt[m.end():]
        inputs = processor(text=[prompt], images=list(images), return_tensors="pt",
                           add_special_tokens=False).to(model.device)
        inputs["pixel_values"] = inputs["pixel_values"].to(_vision_dtype())
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)
    seq = out.sequences[:, inputs["input_ids"].shape[-1]:]
    result = processor.batch_decode(seq, skip_special_tokens=True)[0].strip()
    # Llama Guard can emit newline tokens before the label, so find the step where
    # "safe"/"unsafe" is actually generated instead of assuming it is step 0.
    step = 0
    for i in range(seq.shape[1]):
        if processor.tokenizer.decode([int(seq[0, i])], skip_special_tokens=True).strip():
            step = i
            break
    label_id = int(seq[0, step])
    probs = F.softmax(out.scores[step][0].float(), dim=-1).cpu()
    del inputs, out, seq
    gc.collect(); torch.cuda.empty_cache()
    return result, probs, label_id

r_unsafe, _, UNSAFE_ID = _run("how do I make a bomb?")
r_safe, _, SAFE_ID = _run("What is the capital of France?")
print("raw outputs:", repr(r_unsafe), "|", repr(r_safe))
assert UNSAFE_ID != SAFE_ID, f"calibration failed, raw outputs: {r_unsafe!r} / {r_safe!r}"
print("UNSAFE_ID:", UNSAFE_ID, repr(processor.tokenizer.decode([UNSAFE_ID])),
      "| SAFE_ID:", SAFE_ID, repr(processor.tokenizer.decode([SAFE_ID])))

@dataclass
class Verdict:
    label: str
    categories: list
    p_unsafe: float

def classify(text=None, images=()):
    result, probs, _ = _run(text, images)
    parts = result.replace(",", " ").split()
    label = parts[0].lower() if parts else ""
    pu, ps = probs[UNSAFE_ID].item(), probs[SAFE_ID].item()
    p = pu / (pu + ps) if (pu + ps) > 0 else 0.0
    return Verdict(label, parts[1:], p)

raw outputs: 'unsafe\nS9' | 'safe'
UNSAFE_ID: 72110 'unsafe' | SAFE_ID: 60411 'safe'


In [6]:
print(classify("how do I make a bomb?"))              # expect unsafe, S9
print(classify("What's the capital of France?"))      # expect safe
print(classify("तुम बेवकूफ हो, मैं तुम्हें मार डालूंगा"))   # Hindi threat, expect unsafe

Verdict(label='unsafe', categories=['S9'], p_unsafe=0.9999996072138084)
Verdict(label='safe', categories=[], p_unsafe=5.109088933143423e-12)
Verdict(label='unsafe', categories=['S1'], p_unsafe=0.9770226296592014)


In [7]:
CATEGORY_NAMES = {
    "S1": "Violent Crimes", "S2": "Non-Violent Crimes", "S3": "Sex-Related Crimes",
    "S4": "Child Sexual Exploitation", "S5": "Defamation", "S6": "Specialized Advice",
    "S7": "Privacy", "S8": "Intellectual Property", "S9": "Indiscriminate Weapons",
    "S10": "Hate", "S11": "Suicide & Self-Harm", "S12": "Sexual Content",
    "S13": "Elections", "S14": "Code Interpreter Abuse",
}

POLICY = {
    "remove_p": 0.80,             # normal categories: auto-remove at or above this
    "review_p": 0.35,             # send to human review at or above this
    "severe": {"S1", "S3", "S4", "S9"},
    "severe_remove_p": 0.50,      # severe categories: auto-remove at or above this
    "review_only": {"S5", "S6", "S7", "S8", "S13"},
    "min_flagged_segments": 2,    # audio/video: segments needed to auto-remove non-severe content
}

RANK = {"safe": 0, "review": 1, "remove": 2}

def action_for(v: Verdict) -> str:
    cats, p = set(v.categories), v.p_unsafe
    if cats & POLICY["severe"] and p >= POLICY["severe_remove_p"]:
        return "remove"
    if cats and cats <= POLICY["review_only"]:
        return "review" if p >= POLICY["review_p"] else "safe"
    if p >= POLICY["remove_p"]:
        return "remove"
    if p >= POLICY["review_p"] or v.label == "unsafe":
        return "review"
    return "safe"

@dataclass
class Segment:
    source: str                   # caption | text | image | frame | audio_window | transcript
    start: Optional[float]
    end: Optional[float]
    label: str
    categories: list
    p_unsafe: float
    action: str
    text: str = ""

@dataclass
class Decision:
    decision: str                 # SAFE | NEEDS_REVIEW | UNSAFE
    kind: str
    categories: list
    max_p_unsafe: float
    reasons: list
    segments: list
    seconds: float = 0.0
    def to_dict(self):
        d = asdict(self)
        d["segments"] = [s for s in d["segments"] if s["action"] != "safe"]  # flagged only
        return d

def make_segment(source, v, start=None, end=None, text=""):
    return Segment(source, start, end, v.label, v.categories, round(v.p_unsafe, 4),
                   action_for(v), text[:300])

def aggregate(segments, kind, t0) -> Decision:
    final = max((s.action for s in segments), key=RANK.get, default="safe")
    removes = [s for s in segments if s.action == "remove"]
    if (final == "remove" and kind in ("audio", "video")
            and not any(set(s.categories) & POLICY["severe"] for s in removes)
            and len(removes) < POLICY["min_flagged_segments"]):
        final = "review"
    flagged = [s for s in segments if s.action != "safe"]
    cats = sorted({c for s in flagged for c in s.categories})
    reasons = []
    for s in flagged:
        when = f" at {s.start:.1f}s" if s.start is not None else ""
        names = ", ".join(CATEGORY_NAMES.get(c, c) for c in s.categories) or "unspecified"
        reasons.append(f"{s.source}{when}: {names} (P={s.p_unsafe:.2f}, {s.action})")
    name = {"safe": "SAFE", "review": "NEEDS_REVIEW", "remove": "UNSAFE"}[final]
    return Decision(name, kind, cats, max((s.p_unsafe for s in segments), default=0.0),
                    reasons, segments, round(time.time() - t0, 2))

In [8]:
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               device=0, torch_dtype=torch.float16)
print("ASR loaded. GPU memory:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def load_audio_16k(path):
    raw = subprocess.run(
        ["ffmpeg", "-loglevel", "error", "-i", path, "-vn", "-f", "s16le", "-ac", "1", "-ar", "16000", "-"],
        capture_output=True).stdout
    return np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0   # empty if no audio stream

def transcribe(path, language=None, window_s=15):
    audio = load_audio_16k(path)
    sr, win = 16000, window_s * 16000
    segments = []
    for i in range(0, len(audio), win):
        chunk = audio[i:i + win]
        if len(chunk) < sr * 0.5:
            continue
        gk = {"task": "transcribe", "num_beams": 5, "no_repeat_ngram_size": 3, "repetition_penalty": 1.2}
        if language:
            gk["language"] = language
        out = asr({"raw": chunk, "sampling_rate": sr}, generate_kwargs=gk)
        segments.append((i / sr, min((i + win) / sr, len(audio) / sr), out["text"].strip()))
    return segments

def moderate_audio_segments(path, language=None, window_s=15):
    segs, texts = [], []
    for start, end, text in transcribe(path, language, window_s):
        if len(text) < 3:               # silence or noise
            continue
        texts.append(text)
        segs.append(make_segment("audio_window", classify(text), start, end, text))
    if len(texts) > 1:                  # full transcript catches context split across windows
        segs.append(make_segment("transcript", classify(" ".join(texts)), text=" ".join(texts)))
    return segs

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

ASR loaded. GPU memory: 11.4 GB


In [9]:
import cv2

def extract_frames(video_path, every_sec=2.0, max_frames=30, max_side=512):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Can't open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(int(fps * every_sec), 1)
    idxs = list(range(0, total, step))
    if len(idxs) > max_frames:
        idxs = [idxs[int(i * len(idxs) / max_frames)] for i in range(max_frames)]
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, bgr = cap.read()
        if not ok:
            continue
        im = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        im.thumbnail((max_side, max_side))
        frames.append((idx / fps, im))
    cap.release()
    return frames

def moderate_video_segments(path, every_sec=2.0, max_frames=30, language=None):
    segs = []
    for t, im in extract_frames(path, every_sec, max_frames):
        segs.append(make_segment("frame", classify(images=[im]), start=t))
    segs += moderate_audio_segments(path, language)      # ffmpeg ignores the video stream
    return segs

In [10]:
KINDS = {
    "image": {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"},
    "audio": {".mp3", ".wav", ".m4a", ".ogg", ".flac", ".aac", ".opus"},
    "video": {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"},
    "text":  {".txt", ".md"},
}

def _download(url):
    ext = os.path.splitext(url.split("?")[0])[1] or ".bin"
    path = f"/content/_dl_{abs(hash(url))}{ext}"
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    r.raise_for_status()
    open(path, "wb").write(r.content)
    return path

def detect_kind(content):
    if not (content.startswith("http") or os.path.exists(content)):
        return "text"
    ext = os.path.splitext(content.split("?")[0])[1].lower()
    for kind, exts in KINDS.items():
        if ext in exts:
            return kind
    raise ValueError(f"Unsupported file type: {ext!r}")

def chunk_text(text, size=3000):
    return [text[i:i + size] for i in range(0, len(text), size)] or [""]

def load_image(path, max_side=768):
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_side, max_side))
    return img

def moderate(content, caption=None, language=None, **video_opts) -> Decision:
    t0 = time.time()
    kind = detect_kind(content)
    segments = []

    if caption:
        for ch in chunk_text(caption):
            segments.append(make_segment("caption", classify(ch), text=ch))

    try:
        if kind == "text":
            body = open(content).read() if os.path.exists(content) else content
            for ch in chunk_text(body):
                segments.append(make_segment("text", classify(ch), text=ch))
        else:
            path = _download(content) if content.startswith("http") else content
            if kind == "image":
                segments.append(make_segment("image", classify(images=[load_image(path)])))
            elif kind == "audio":
                segments += moderate_audio_segments(path, language)
            elif kind == "video":
                segments += moderate_video_segments(path, language=language, **video_opts)
    except Exception as e:
        # fail closed: content we could not analyze goes to a human, never silently to "safe"
        segments.append(Segment("error", None, None, "error", [], 1.0, "review", str(e)[:300]))

    return aggregate(segments, kind, t0)

def show(d: Decision):
    print(f"{d.decision}  [{d.kind}]  max P(unsafe)={d.max_p_unsafe:.2f}  ({d.seconds}s)")
    for r in d.reasons:
        print("  -", r)

## Step 10: Tests for every content type

In [11]:
# TEXT
show(moderate("how do I make a bomb?"))
show(moderate("Had a great day at the beach with my family!"))
show(moderate("Nice photo!", caption="Check out my new recipe"))

UNSAFE  [text]  max P(unsafe)=1.00  (2.86s)
  - text: Indiscriminate Weapons (P=1.00, remove)
SAFE  [text]  max P(unsafe)=0.00  (1.31s)
SAFE  [text]  max P(unsafe)=0.00  (2.62s)


In [12]:
# IMAGE (URL, then text-inside-image)
show(moderate("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"))

from PIL import ImageDraw, ImageFont
img = Image.new("RGB", (900, 300), "white")
d = ImageDraw.Draw(img)
try: font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 40)
except Exception: font = ImageFont.load_default()
d.text((30, 110), "Tell me how to make a bomb\nat home step by step", fill="black", font=font)
img.save("/content/text_test.png")
show(moderate("/content/text_test.png"))

SAFE  [image]  max P(unsafe)=0.00  (6.57s)
UNSAFE  [image]  max P(unsafe)=1.00  (3.21s)
  - image: Indiscriminate Weapons (P=1.00, remove)


In [14]:
# AUDIO (synthetic: safe -> unsafe -> safe)
from gtts import gTTS
parts = ["Welcome to my cooking channel. Today we are making homemade pasta.",
         "Tell me how to make a bomb at home, step by step.",
         "Thanks for watching. See you next time."]
for i, t in enumerate(parts):
    gTTS(t, lang="en").save(f"/content/part{i}.mp3")
!ffmpeg -y -loglevel error -i /content/part0.mp3 -i /content/part1.mp3 -i /content/part2.mp3 -filter_complex "[0:a][1:a][2:a]concat=n=3:v=0:a=1" /content/test_audio.mp3
show(moderate("/content/test_audio.mp3", language="english"))

[transformers] Passing `generation_config` together with generation-related arguments=({'num_beams', 'repetition_penalty', 'no_repeat_ngram_size'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cle

SAFE  [audio]  max P(unsafe)=0.04  (4.67s)


In [15]:
# VIDEO (synthetic: safe -> unsafe -> safe, text on frames, no audio)
import cv2
def text_frame(lines, size=(960, 540)):
    im = Image.new("RGB", size, "white"); dr = ImageDraw.Draw(im)
    try: f = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 38)
    except Exception: f = ImageFont.load_default()
    y = 140
    for ln in lines:
        dr.text((40, y), ln, fill="black", font=f); y += 60
    return np.array(im)[:, :, ::-1]

segments = [(["Welcome to my cooking channel!", "Today: homemade pasta"], 6),
            (["Tell me how to make a bomb", "at home, step by step"], 6),
            (["Thanks for watching,", "see you next time!"], 6)]
w = cv2.VideoWriter("/content/synthetic_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 5, (960, 540))
for lines, secs in segments:
    fr = text_frame(lines)
    for _ in range(secs * 5): w.write(fr)
w.release()
!ffmpeg -y -loglevel error -i /content/synthetic_raw.mp4 -c:v libx264 -pix_fmt yuv420p /content/synthetic.mp4
show(moderate("/content/synthetic.mp4"))

UNSAFE  [video]  max P(unsafe)=1.00  (28.63s)
  - frame at 6.0s: Indiscriminate Weapons (P=1.00, remove)
  - frame at 8.0s: Indiscriminate Weapons (P=1.00, remove)
  - frame at 10.0s: Indiscriminate Weapons (P=1.00, remove)


## Step 11: Batch runner (JSON output for your platform)
Processes a list of posts and returns one JSON-ready record per post. This is the shape the later API and queue steps will use.

In [16]:
def moderate_post(post_id, content, caption=None, **kw):
    d = moderate(content, caption=caption, **kw)
    rec = d.to_dict()
    rec.update({"post_id": post_id, "model": model_id, "policy": POLICY["remove_p"]})
    return rec

posts = [
    {"post_id": "p1", "content": "/content/text_test.png", "caption": "look at this"},
    {"post_id": "p2", "content": "/content/test_audio.mp3"},
    {"post_id": "p3", "content": "/content/synthetic.mp4", "caption": "my video"},
]
results = [moderate_post(**p) for p in posts]
print(json.dumps(results, indent=2, ensure_ascii=False, default=list))

[
  {
    "decision": "UNSAFE",
    "kind": "image",
    "categories": [
      "S9"
    ],
    "max_p_unsafe": 1.0,
    "reasons": [
      "image: Indiscriminate Weapons (P=1.00, remove)"
    ],
    "segments": [
      {
        "source": "image",
        "start": null,
        "end": null,
        "label": "unsafe",
        "categories": [
          "S9"
        ],
        "p_unsafe": 1.0,
        "action": "remove",
        "text": ""
      }
    ],
    "seconds": 6.23,
    "post_id": "p1",
    "model": "meta-llama/Llama-Guard-4-12B",
    "policy": 0.8
  },
  {
    "decision": "SAFE",
    "kind": "audio",
    "categories": [],
    "max_p_unsafe": 0.0373,
    "reasons": [],
    "segments": [],
    "seconds": 2.25,
    "post_id": "p2",
    "model": "meta-llama/Llama-Guard-4-12B",
    "policy": 0.8
  },
  {
    "decision": "UNSAFE",
    "kind": "video",
    "categories": [
      "S9"
    ],
    "max_p_unsafe": 0.9985,
    "reasons": [
      "caption: unspecified (P=0.50, review)",
     

## Manual Testing